In [1]:
# whitespace, переносов нет. тире/дефис, лапки, гомоглифu
# each row is one semantic unit, so no splitting

In [2]:
import pandas as pd
import re
import unicodedata

# Display settings
pd.set_option('display.expand_frame_repr', False)

# Load CSV files
df = pd.read_csv('processed.csv', encoding='utf-8')
print(df.head(15))
labels = pd.read_csv('labels.csv', encoding='utf-8')

# ---------- TEXT CLEANING FUNCTION ----------

def clean_text(text):
    if pd.isna(text):
        return text

    # Unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text.strip())

    # Unify apostrophes
    text = re.sub(r"[‘’`]", "'", text)

    # Unify quotation marks
    text = re.sub(r'[“”«»„]', '"', text)

    # Normalize dashes (hyphen, en dash, em dash, minus)
    text = re.sub(r'[–—−]', '-', text)

    return text


# Apply cleaning
df['statement_1'] = df['statement_1'].apply(clean_text)
df['statement_2'] = df['statement_2'].apply(clean_text)

# ---------- VALIDATION / CHECKING ----------

def contains_cyrillic(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[А-Яа-яЁёІіЇїЄє]', text))


def contains_latin(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[A-Za-z]', text))


def mixed_alphabet(text):
    if pd.isna(text):
        return False
    return contains_latin(text) and contains_cyrillic(text)


def find_special_dashes(text):
    if pd.isna(text):
        return []
    return re.findall(r'[–—−]', text)


def find_special_quotes(text):
    if pd.isna(text):
        return []
    return re.findall(r'[“”«»„]', text)



df['has_cyrillic_1'] = df['statement_1'].apply(contains_cyrillic)
df['has_cyrillic_2'] = df['statement_2'].apply(contains_cyrillic)

df['mixed_alphabet_1'] = df['statement_1'].apply(mixed_alphabet)
df['mixed_alphabet_2'] = df['statement_2'].apply(mixed_alphabet)

df['special_dashes_1'] = df['statement_1'].apply(find_special_dashes)
df['special_dashes_2'] = df['statement_2'].apply(find_special_dashes)

df['special_quotes_1'] = df['statement_1'].apply(find_special_quotes)
df['special_quotes_2'] = df['statement_2'].apply(find_special_quotes)


df.to_csv('processed_cleaned.csv', index=False, encoding='utf-8')

df.head()

                    statement_1                       statement_2
0            Що? Я не чую тебе.                Що? Я тебе не чую.
1                   Вони щезли.     Він не міг зробити подібного.
2       В тебе було повно часу.          В тебе було багато часу.
3                    Я товстий.                        Я гладкий.
4         В мене багато друзів.               Вечірка скінчилася.
5       Я живу тут з 1990 року.              Це японський прапор.
6              Вже майже третя.           Я зі Сполучених Штатів.
7            У нас є білий кіт.                    Де ти мешкаєш?
8               Том має роботу.                  У Тома є робота.
9             Засвітили світло.                    Що це значить?
10         В мене немає грошей.              У мене немає грошей.
11          Я живу в Білостоці.                        Хтозна що.
12           Хуан не має брата.              У Хуана немає брата.
13  Мій брат зараз в Австралії.           Брат зараз в Австралії.
14  Я зазв

,statement_1,statement_2,has_cyrillic_1,has_cyrillic_2,mixed_alphabet_1,mixed_alphabet_2,special_dashes_1,special_dashes_2,special_quotes_1,special_quotes_2
0,Що? Я не чую тебе.,Що? Я тебе не чую.,True,True,False,False,[],[],[],[]
1,Вони щезли.,Він не міг зробити подібного.,True,True,False,False,[],[],[],[]
2,В тебе було повно часу.,В тебе було багато часу.,True,True,False,False,[],[],[],[]
3,Я товстий.,Я гладкий.,True,True,False,False,[],[],[],[]
4,В мене багато друзів.,Вечірка скінчилася.,True,True,False,False,[],[],[],[]


In [3]:
# Different alphabets (mixed Latin + Cyrillic)
mixed_alphabet_count = (
    df['mixed_alphabet_1'] | df['mixed_alphabet_2']
).sum()

print("Rows with mixed alphabets:", mixed_alphabet_count)


# Cyrillic presence
cyrillic_count = (
    df['has_cyrillic_1'] | df['has_cyrillic_2']
).sum()

print("Rows with Cyrillic characters:", cyrillic_count)


# Special dashes
special_dashes_count = (
    (df['special_dashes_1'].str.len() > 0) |
    (df['special_dashes_2'].str.len() > 0)
).sum()

print("Rows with special dashes:", special_dashes_count)


# Special quotes
special_quotes_count = (
    (df['special_quotes_1'].str.len() > 0) |
    (df['special_quotes_2'].str.len() > 0)
).sum()

print("Rows with special quotes:", special_quotes_count)
df.info()

Rows with mixed alphabets: 24
Rows with Cyrillic characters: 2502
Rows with special dashes: 0
Rows with special quotes: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2502 entries, 0 to 2501
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   statement_1       2502 non-null   object
 1   statement_2       2502 non-null   object
 2   has_cyrillic_1    2502 non-null   bool  
 3   has_cyrillic_2    2502 non-null   bool  
 4   mixed_alphabet_1  2502 non-null   bool  
 5   mixed_alphabet_2  2502 non-null   bool  
 6   special_dashes_1  2502 non-null   object
 7   special_dashes_2  2502 non-null   object
 8   special_quotes_1  2502 non-null   object
 9   special_quotes_2  2502 non-null   object
dtypes: bool(4), object(6)
memory usage: 127.2+ KB


In [4]:
# Filter mixed alphabet rows
mixed_rows = df[df['mixed_alphabet_1'] | df['mixed_alphabet_2']]

# Take 20 random records
sample_mixed = mixed_rows.sample(n=20, random_state=42).copy()

# Create required columns
sample_mixed = sample_mixed.reset_index().rename(columns={
    'index': 'id',
    'statement_1': 'raw_text_1',
    'statement_2': 'raw_text_2'
})

# Keep only required columns
sample_mixed = sample_mixed[['id', 'raw_text_1', 'raw_text_2']]

# Add expected behavior column
sample_mixed['expected_behavior'] = 'fix mixed alphabet'

# Save to CSV
sample_mixed.to_csv('edge_cases.csv', index=False)

In [5]:
import re

# Homoglyph mapping
latin_to_cyrillic = str.maketrans({
    'a': 'а', 'e': 'е', 'i': 'і', 'o': 'о',
    'p': 'р', 'c': 'с', 'x': 'х', 'y': 'у',
    'k': 'к', 'm': 'м', 't': 'т', 'b': 'в',
    'h': 'н',
    'A': 'А', 'E': 'Е', 'I': 'І', 'O': 'О',
    'P': 'Р', 'C': 'С', 'X': 'Х', 'Y': 'У',
    'K': 'К', 'M': 'М', 'T': 'Т', 'B': 'В',
    'H': 'Н'
})

# Explicit exceptions based on your dataset
EXCEPTIONS = [
    r'\bpretty\b',                     # English word "pretty"
    r'Алфавіт есперанто',              # Esperanto alphabet rows
    r'a, b, c,',                        # Alphabet list pattern
]

def smart_homoglyph_fix_with_exceptions(text):
    if pd.isna(text):
        return text

    # Skip exceptions
    for pattern in EXCEPTIONS:
        if re.search(pattern, text, re.IGNORECASE):
            return text

    # Check for mixed Cyrillic + Latin
    has_cyr = bool(re.search(r'[А-Яа-яЁёІіЇїЄє]', text))
    has_lat = bool(re.search(r'[A-Za-z]', text))

    if has_cyr and has_lat:
        # Convert isolated Latin homoglyphs to Cyrillic
        return text.translate(latin_to_cyrillic)

    return text

# Apply to both columns
df['statement_1'] = df['statement_1'].apply(smart_homoglyph_fix_with_exceptions)
df['statement_2'] = df['statement_2'].apply(smart_homoglyph_fix_with_exceptions)

# Remove identical rows
df = df[df['statement_1'] != df['statement_2']].reset_index(drop=True)

df['mixed_alphabet_1'] = df['statement_1'].apply(mixed_alphabet)
df['mixed_alphabet_2'] = df['statement_2'].apply(mixed_alphabet)

# Different alphabets (mixed Latin + Cyrillic)
mixed_alphabet_count = (
    df['mixed_alphabet_1'] | df['mixed_alphabet_2']
).sum()

print("Rows with mixed alphabets:", mixed_alphabet_count)
mixed_rows = df[
    df['mixed_alphabet_1'] | df['mixed_alphabet_2']
]
print(mixed_rows)

Rows with mixed alphabets: 8
                                            statement_1                                        statement_2  has_cyrillic_1  has_cyrillic_2  mixed_alphabet_1  mixed_alphabet_2 special_dashes_1 special_dashes_2 special_quotes_1 special_quotes_2
526                                    У неї є піаніно?                        Як пишеться слово "pretty"?            True            True             False              True               []               []               []               []
765                               Як пишеться "pretty"?                                        Знов осінь.            True            True              True             False               []               []               []               []
896                               Як пишеться "pretty"?                       Сподіваюся, тобі не страшно.            True            True              True             False               []               []               []               []

In [6]:
df = df[['statement_1', 'statement_2']].copy()
df.head(15)

,statement_1,statement_2
0,Що? Я не чую тебе.,Що? Я тебе не чую.
1,Вони щезли.,Він не міг зробити подібного.
2,В тебе було повно часу.,В тебе було багато часу.
3,Я товстий.,Я гладкий.
4,В мене багато друзів.,Вечірка скінчилася.
5,Я живу тут з 1990 року.,Це японський прапор.
6,Вже майже третя.,Я зі Сполучених Штатів.
7,У нас є білий кіт.,Де ти мешкаєш?
8,Том має роботу.,У Тома є робота.
9,Засвітили світло.,Що це значить?


In [7]:
df.to_csv('processed_v2.csv', index=False)

sample_df = df.sample(frac=0.10, random_state=42)
sample_df.to_csv('sample.csv', index=False)